# Laboratorio RAG - NLP Research Assistant

Sistema completo di Retrieval-Augmented Generation su articoli scientifici NLP.

**Obiettivo**: Costruire un assistente che risponde a domande sulla letteratura NLP citando fonti.

**Dataset**: `MaartenGr/arxiv_nlp` (abstract di articoli da arXiv)

**LLM**: Qwen2.5-0.5B (500MB, locale, gratis)

## 1. Installazione Dipendenze

In [ ]:
%%capture
!pip install datasets transformers sentence-transformers faiss-cpu torch accelerate

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import faiss

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. Caricamento Dataset

In [ ]:
print("Caricamento dataset arXiv NLP...")
dataset = load_dataset("MaartenGr/arxiv_nlp", split="train")
df = pd.DataFrame(dataset)

# Usa subset per velocità (rimuovi .head() per dataset completo)
df = df.head(500)

print(f"Documenti caricati: {len(df)}")
print(f"Colonne: {df.columns.tolist()}")
df.head(2)

In [ ]:
# Prepara documenti
documents = []
metadata = []

# Auto-rileva colonne
title_col = next((c for c in df.columns if 'title' in c.lower()), df.columns[0])
abstract_col = next((c for c in df.columns if 'abstract' in c.lower() or 'summar' in c.lower()), df.columns[1])

print(f"Colonna titolo: {title_col}")
print(f"Colonna abstract: {abstract_col}\n")

for idx, row in df.iterrows():
    title = str(row[title_col]) if pd.notna(row[title_col]) else 'No Title'
    abstract = str(row[abstract_col]) if pd.notna(row[abstract_col]) else 'No Abstract'
    
    documents.append(f"Title: {title}\n\nAbstract: {abstract}")
    metadata.append({
        'title': title,
        'year': str(row.get('year', 'Unknown'))
    })

print(f"Documenti preparati: {len(documents)}")
print(f"\nEsempio:\n{documents[0][:300]}...")

## 3. Generazione Embeddings

In [ ]:
print("Caricamento modello di embedding...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generazione embeddings...")
embeddings = embed_model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")

## 4. Vector Store (FAISS)

In [ ]:
# Crea indice FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product (cosine similarity)
index.add(embeddings)

print(f"Indice FAISS creato: {index.ntotal} vettori")

## 5. Retriever

In [ ]:
def retrieve(query, k=5):
    """Recupera i k documenti più rilevanti"""
    query_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    similarities, indices = index.search(query_emb, k)
    
    results = []
    for i, (sim, idx) in enumerate(zip(similarities[0], indices[0])):
        results.append({
            'rank': i + 1,
            'similarity': float(sim),
            'document': documents[idx],
            'metadata': metadata[idx]
        })
    return results

# Test retrieval
test_query = "What are transformer models?"
results = retrieve(test_query, k=3)

print(f"Query: {test_query}\n")
for r in results:
    print(f"[{r['rank']}] Similarity: {r['similarity']:.3f}")
    print(f"    {r['metadata']['title'][:80]}")
    print()

## 6. LLM Locale (Qwen 0.5B)

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Caricamento {model_id}...")
print("Download: ~500MB, tempo: ~1-2 minuti\n")

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
)

llm = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1,
    max_new_tokens=300,
)

print(f"Modello caricato su {device}")

## 7. Sistema RAG Completo

In [ ]:
def rag_answer(query, k=3, verbose=True):
    """Risponde alla query usando RAG"""
    
    if verbose:
        print(f"Query: {query}\n")
        print("=" * 60)
        print("RETRIEVAL")
        print("=" * 60)
    
    # 1. Retrieval
    results = retrieve(query, k=k)
    
    if verbose:
        for r in results:
            print(f"[{r['rank']}] {r['metadata']['title'][:70]}")
            print(f"    Similarity: {r['similarity']:.3f}")
    
    # 2. Costruisci contesto
    context_parts = []
    for r in results:
        context_parts.append(f"[Document {r['rank']}]\n{r['document']}")
    
    context = "\n\n".join(context_parts)
    
    # Limita contesto per modelli piccoli
    if len(context) > 1500:
        context = context[:1500] + "\n[...truncated...]"
    
    # 3. Costruisci prompt
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant specialized in NLP research. Answer based on the provided context."
        },
        {
            "role": "user",
            "content": f"""Context from research papers:

{context}

Question: {query}

Answer based on the context above:"""
        }
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # 4. Genera risposta
    if verbose:
        print("\n" + "=" * 60)
        print("GENERATION")
        print("=" * 60)
    
    output = llm(
        prompt,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Estrai risposta
    full_text = output[0]['generated_text']
    answer = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text
    
    if verbose:
        print("\n" + "=" * 60)
        print("ANSWER")
        print("=" * 60)
        print(answer)
        print("\n" + "=" * 60)
        print("SOURCES")
        print("=" * 60)
        for r in results:
            print(f"- {r['metadata']['title']} ({r['metadata']['year']})")
    
    return {
        'query': query,
        'answer': answer,
        'sources': results
    }

## 8. Test del Sistema

In [ ]:
result = rag_answer("What is BERT and how does it work?", k=3)

In [ ]:
result = rag_answer("Explain the attention mechanism in transformers", k=3)

In [ ]:
result = rag_answer("What are the main differences between BERT and GPT?", k=3)

## 9. Query Personalizzate

In [ ]:
# Prova le tue query qui
my_query = "What is transfer learning in NLP?"
result = rag_answer(my_query, k=3)

## 10. Analisi Retrieval

In [ ]:
# Analizza distribuzione similarità
import matplotlib.pyplot as plt

query = "What are neural networks?"
results = retrieve(query, k=20)

similarities = [r['similarity'] for r in results]
ranks = [r['rank'] for r in results]

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(ranks, similarities, 'o-', color='#4A90E2')
plt.xlabel('Rank')
plt.ylabel('Similarity')
plt.title('Similarity vs Rank')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(similarities, bins=15, color='#4A90E2', alpha=0.7, edgecolor='black')
plt.xlabel('Similarity')
plt.ylabel('Frequency')
plt.title('Distribution of Similarities')

plt.tight_layout()
plt.show()

print(f"Query: {query}")
print(f"Mean similarity: {np.mean(similarities):.3f}")
print(f"Std similarity: {np.std(similarities):.3f}")

## Esercizi

1. **Aumenta dataset**: Rimuovi `df.head(500)` per usare l'intero dataset

2. **Hybrid search**: Implementa combinazione BM25 + embeddings
   ```python
   from rank_bm25 import BM25Okapi
   # Implementa funzione hybrid_retrieve()
   ```

3. **Re-ranking**: Usa cross-encoder per migliorare precision
   ```python
   from sentence_transformers import CrossEncoder
   reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
   ```

4. **Evaluation**: Crea test set e misura Precision@k

5. **Temperature tuning**: Sperimenta con temperature diverse (0.1-0.9)

6. **Modello migliore**: Sostituisci Qwen con Phi-2 per qualità superiore
   ```python
   model_id = "microsoft/phi-2"
   ```

## Risorse

**Paper**:
- Lewis et al. (2020). Retrieval-Augmented Generation. arXiv:2005.11401
- Karpukhin et al. (2020). Dense Passage Retrieval. arXiv:2004.04906

**Documentazione**:
- HuggingFace Transformers: https://huggingface.co/docs/transformers
- Sentence Transformers: https://www.sbert.net
- FAISS: https://github.com/facebookresearch/faiss

**Modelli alternativi**:
- SmolLM-360M: più veloce (~300MB)
- Phi-2: migliore qualità (~5GB)
- GPT-3.5 (API): qualità superiore ma richiede API key